In [16]:
import os
import json
import time
import pandas as pd
from openai import OpenAI
from tqdm import tqdm

# ========== 1. 配置 ==========
API_KEY = ""
MODEL_NAME = "qwen-plus"
INPUT_FILE = "MixedTypeData1.csv"
OUTPUT_FILE = "tagged_with_timeseries.csv"
CACHE_FILE = "timeseries_cache2.json"

# ========== 测试模式参数 ==========
START_ROW_INDEX = None      # 从第几行开始，None表示从头
NUM_SAMPLES = None          # 处理多少条，None表示到末尾
IGNORE_CACHE_FOR_TEST = False

# ========== 2. 初始化客户端 ==========
client = OpenAI(
    api_key=API_KEY,
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)

# ========== 3. 标签列映射 ==========
LABEL_COLUMNS = {
    'G1': 'G1', 'G2': 'G2', 'G3': 'G3',
    'L1': 'L1', 'L2': 'L2', 'L3': 'L3',
    'R1': 'R1', 'R2': 'R2', 'R3': 'R3'
}

def extract_existing_labels(row):
    labels = []
    for col, code in LABEL_COLUMNS.items():
        if col in row and row[col] == 1:
            labels.append(code)
    return labels

# ========== 4. 构建 Prompt ==========
def build_prompt(query, existing_labels, idx):
    return f"""已知以下文本已被标注为包含下列标签（标签代码见末尾），请根据文本中这些手法出现的自然顺序，输出时间序列（只对已有标签排序）。

文本：{query}
已有标签：{existing_labels}  

规则：
1.按手法在文本中出现的先后顺序排列，最早出现的排第一。例如 ["G1","R2"]
2.若两个手法在同一句话或同一个紧密耦合的表述中同时出现（如“我是公安局的，不转账就冻结账户”同时包含 R2 和 L2），用 & 连接，例如 ["R2 & L2", "G1"]。
3.若顺序无法判断（如信息不足或手法交错复杂），输出 ["unknown"]。
4.G1包括法律赔偿、退款、和解金等任何直接金钱利益，不限于“刷单返利”等简单表述。
5.顺序必须基于诈骗实施的自然流程，而不是报道中提及的顺序。比如冒充通常发生在利益承诺之前。
6.禁止重复与冗余：每个原始标签在时间序列中只能出现一次。如果一个标签已经被包含在 & 组合中（如 R2 & L2 已经包含了 R2 和 L2），则不能再单独列出该标签（即不能再输出单独的 "R2" 或 "L2"）。最终输出的序列中，所有 & 内外的原子标签去重后，必须与输入的已有标签集合完全相等，不能多也不能少。

标签代码解释：
- G1:财富诱饵，G2:关系诱饵，G3:机会诱饵
- L1:安全威胁，L2:财产威胁，L3:隐私威胁
- R1:亲友冒充，R2:权威冒充，R3:客服/平台冒充

输出格式（严格按模板，不得修改结构）：
---
样本ID：{idx}
时间序列：[]
OT描述：
---

请直接输出，不要添加任何额外解释。"""

def call_qwen(query, existing_labels, idx):
    system_prompt = "你是严谨的诈骗文本标注助手，严格按规则判断时间序列，输出简体中文，不主观发挥。"
    user_prompt = build_prompt(query, existing_labels, idx)
    
    max_retries = 3
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0,
                timeout=60
            )
            raw_output = response.choices[0].message.content
            for line in raw_output.strip().split('\n'):
                if '时间序列：' in line:
                    return line.split('时间序列：')[1].strip()
            return ""
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"\n样本ID {idx} 重试 {max_retries} 次仍失败: {e}")
                return ""
            time.sleep(2 ** attempt)   # 指数退避
    return ""

# ========== 5. 主程序（增强版：实时监控 + 周期性保存CSV） ==========
def main():
    # 加载缓存
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, 'r', encoding='utf-8') as f:
            cache = json.load(f)
        print(f"已加载缓存 {len(cache)} 条")
    else:
        cache = {}
    
    # 读取数据
    if INPUT_FILE.endswith('.csv'):
        df = pd.read_csv(INPUT_FILE)
    else:
        df = pd.read_excel(INPUT_FILE)
    print(f"原始数据共 {len(df)} 行")
    
    # 确保有文本列
    if 'content' not in df.columns:
        raise KeyError("表格中缺少 'content' 列，请修改代码中的列名")
    
    # 确定ID列
    if '样本ID' not in df.columns:
        df['样本ID'] = df.index
        print("未找到 '样本ID' 列，将使用行索引作为样本ID")
    
    # 确定处理范围（按行号）
    if START_ROW_INDEX is not None:
        start_idx = START_ROW_INDEX
    else:
        start_idx = 0
    
    if NUM_SAMPLES is not None:
        end_idx = min(start_idx + NUM_SAMPLES, len(df))
    else:
        end_idx = len(df)
    
    df_subset = df.iloc[start_idx:end_idx].copy()
    print(f"处理范围：行索引 {start_idx} 到 {end_idx-1}，共 {len(df_subset)} 条")
    
    # 准备结果列
    if 'time_series' not in df.columns:
        df['time_series'] = ""
    
    # ========== 循环处理（增强版） ==========
    success_count = 0
    save_interval = 50   # 每处理50条保存一次CSV（可根据需要调整）
    
    for idx_in_subset, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc="打标进度"):
        sample_id = str(row['样本ID'])
        
        # 检查缓存
        if not IGNORE_CACHE_FOR_TEST and sample_id in cache:
            df.at[idx_in_subset, 'time_series'] = cache[sample_id]
            if cache[sample_id] and cache[sample_id] != "":
                success_count += 1
            continue
        
        # 处理新数据
        query = row['content']
        if pd.isna(query) or query == "":
            result = "[]"
        else:
            existing_labels = extract_existing_labels(row)
            if not existing_labels:
                result = "[]"
            else:
                result = call_qwen(query, existing_labels, sample_id)
        
        # 保存结果
        df.at[idx_in_subset, 'time_series'] = result
        cache[sample_id] = result
        if result and result != "":
            success_count += 1
        
        # 实时保存缓存（每一条都存）
        with open(CACHE_FILE, 'w', encoding='utf-8') as f:
            json.dump(cache, f, ensure_ascii=False, indent=2)
        
        # 周期性保存 CSV（避免中断后丢失所有进度）
        if (idx_in_subset - start_idx + 1) % save_interval == 0:
            df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
            print(f"\n[进度] 已处理 {(idx_in_subset - start_idx + 1)} 条，成功 {success_count} 条，中间结果已保存")
    
    # 最终保存一次 CSV
    df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    
    # 最终统计
    total_processed = len(df_subset)
    print(f"\n处理完成！共处理 {total_processed} 条，成功写入非空结果 {success_count} 条")
    print(f"结果已保存至 {OUTPUT_FILE}")
    print(f"缓存文件：{CACHE_FILE}")

if __name__ == "__main__":
    main()

已加载缓存 6 条
原始数据共 5 行
处理范围：行索引 0 到 4，共 5 条


打标进度: 100%|███████████████████████████████████| 5/5 [00:26<00:00,  5.27s/it]


处理完成！共处理 5 条，成功写入非空结果 5 条
结果已保存至 tagged_with_timeseries2.csv
缓存文件：timeseries_cache2.json
